# Assignment 1 — QANet

**COMP5329 / Deep Learning — University of Sydney, Semester 1 2026**

Run each section in order. Sections 0-2 are setup and preprocessing; the remaining sections are controlled ablation experiments on the QANet training pipeline.

Each experiment isolates one design choice while keeping the rest of the training configuration fixed, so the resulting differences can be interpreted as evidence about that specific factor rather than a side effect of changing multiple hyperparameters at once.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust this path if your repo is stored elsewhere in Drive.
PROJECT_ROOT = "/content/drive/MyDrive/Assignment1_2026"

In [ ]:
# Install Python dependencies (run once per session)
!pip install -r {PROJECT_ROOT}/requirements.txt -q
!python -m spacy download en

---
## Section 0 — Environment Setup

Mount Google Drive and install dependencies.

In [ ]:
import sys, os

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

---
## Section 1 — Download Data *(delete before submitting)*

Downloads the pre-built mini dataset (sampled SQuAD v1.1 train + full dev set,
with GloVe vectors filtered to the mini vocabulary) from GitHub Releases into `_data/`.

> **One-time step.** Once `_data/` exists on your Drive, delete this section before submission.

In [ ]:
from Tools.download import download_mini

download_mini(data_dir="_data")

---
## Section 2 — Preprocess Data *(delete before submitting)*

Tokenises the SQuAD JSON files, builds word/char vocabularies from GloVe, and writes padded index tensors to `_data/`.

> **One-time step.** Once `_data/*.npz` exists on your Drive, delete this section before submission. Re-run only if you change `para_limit`, `ques_limit`, or other shape parameters.

In [ ]:
from Tools.preproc import preprocess

preprocess(
    train_file="_data/squad/train-mini.json",
    dev_file="_data/squad/dev-v1.1.json",
    glove_word_file="_data/glove/glove.mini.txt",
    target_dir="_data",
    para_limit=400,
    ques_limit=50,
)

---
## Experiment 1: Effect of Normalization Strategy

This experiment studies whether the choice of normalization changes optimization stability and final question-answering performance. The two conditions are `layer_norm` and `group_norm`, and both are trained with the same data split, training budget, optimizer, scheduler, batch size, loss, and model size.

**Research question.** Under the same training setup, does LayerNorm or GroupNorm lead to faster convergence, smoother optimization, and better final Dev F1 / EM?

**Hypothesis.** LayerNorm is expected to work better for this token-level sequence model because it normalizes features per position and is commonly used in Transformer-style encoders.

**What this cell does.** It runs `run_experiment1.py`, trains both normalization variants, performs official evaluation on the dev set, writes detailed logs/checkpoints under `exp_outputs/experiment1_norm/`, and prints a comparison table summarizing the main metrics.

**What to look at when analysing results.** Compare `dev_f1`, `dev_em`, `dev_loss`, and gradient norm trends across training. In particular, pay attention to which condition reaches useful performance earlier, which one has more stable curves, and which one achieves the best final `official_eval_f1` and `official_eval_em`.

In [ ]:
from run_experiment1 import run_experiment1

exp1_results = run_experiment1()

# Optional: generate plots after the experiment has finished
# from run_experiment1 import plot_experiment1_results
# plot_experiment1_results()


---
## Experiment 2: Effect of Activation Function

This experiment evaluates whether the activation function influences gradient flow, convergence speed, and final QA quality. The compared conditions are `relu` and `leaky_relu`, with optimizer, scheduler, loss, batch size, data, and training budget controlled.

**Research question.** If everything else is fixed, does replacing ReLU with LeakyReLU improve optimization behaviour or final performance in QANet?

**Hypothesis.** LeakyReLU may reduce the risk of dead neurons and therefore produce slightly more stable gradients, especially in a deep architecture with repeated encoder blocks.

**What this cell does.** It runs `run_experiment2.py`, trains both activation variants, performs the official dev-set evaluation, writes outputs to `exp_outputs/experiment2_activation/`, and prints the comparison table.

**What to discuss in the report.** Compare early-stage learning speed, the stability of `dev_loss` and gradient norm, and the final `official_eval_f1` / `official_eval_em`. If the scores are close, it is still useful to discuss whether one activation is easier to optimize or reaches its best checkpoint earlier.

In [ ]:
from run_experiment2 import run_experiment2

activation_results = run_experiment2()

# Optional: generate plots after the experiment has finished
# from run_experiment2 import plot_experiment2_results
# plot_experiment2_results()


---
## Experiment 3: Effect of Weight Initialization

This experiment tests how parameter initialization affects early optimization and final accuracy. The compared conditions are `kaiming` and `xavier`, while activation, optimizer, scheduler, loss, batch size, and training budget are kept fixed.

**Research question.** With the rest of the model and training pipeline unchanged, does Kaiming initialization or Xavier initialization produce better learning dynamics and downstream QA performance?

**Why this matters.** Initialization controls the scale of activations and gradients at the start of training. A poor initialization can slow convergence or make optimization unstable, even if the architecture is otherwise correct.

**What this cell does.** It runs `run_experiment3.py`, trains both initialization variants, evaluates each best checkpoint on the dev set, stores outputs in `exp_outputs/experiment3_init/`, and prints a summary table for direct comparison.

**Suggested interpretation.** Check which condition reduces training loss more smoothly, reaches non-trivial Dev F1 earlier, and produces the highest final official evaluation scores. Since the default activation is ReLU, it is also reasonable to discuss whether Kaiming has an advantage because it is designed to preserve signal scale under ReLU-like nonlinearities.

In [ ]:
from run_experiment3 import run_experiment3

initialization_results = run_experiment3()

# Optional: generate plots after the experiment has finished
# from run_experiment3 import plot_experiment3_results
# plot_experiment3_results()
